# 패턴 인식 기반 파쇄기 이물질 끼임 감지

## 개요
산업용 2축 파쇄기(Shredder)의 **이물질 끼임(Jamming)** 현상을 실시간으로 감지하는 패턴 인식 알고리즘입니다.

### 핵심 원리
- **전류 스파이크 감지**: 이물질 끼임 시 모터 부하 급증 → 전류 급상승
- **속도 드롭 감지**: 부하 증가로 인한 회전 속도 급감 (관성 지연 포함)
- **다중 증거 융합**: 전류 + 속도 + 변화율을 종합하여 오탐 최소화

### 데이터 스펙
| 항목 | 값 |
|------|-----|
| 샘플링 주기 | 10ms (100Hz) |
| 총 관측 시간 | 300초 (5분) |
| 총 샘플 수 | 30,000 |
| 끼임 이벤트 | 11건 (경미/심각/충격) |

### 상태 천이 모델
```
INIT → NORMAL → CAUTION → WARNING → JAMMING
                   ↑          ↑         |
                   └──────────┴─────────┘ (복귀)
```

## Step 0. 라이브러리 임포트

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import deque, OrderedDict
import warnings
warnings.filterwarnings('ignore')

# Matplotlib configuration
plt.rcParams.update({
    'figure.figsize': (16, 5),
    'figure.dpi': 100,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'font.size': 11,
    'axes.titlesize': 14,
    'axes.labelsize': 12,
})

print('=== Shredder Jamming Detection System ===')
print(f'NumPy: {np.__version__}')
print(f'Pandas: {pd.__version__}')
print('Libraries loaded successfully.')

## Step 1. 고주파 데이터 생성 (10ms 샘플링)

실제 2축 파쇄기의 센서 데이터를 시뮬레이션합니다.

### 신호 구성
- **A축 (주축)**: 기준 전류 85A, 회전 리플 20Hz, 기준 속도 1200 RPM
- **B축 (보조축)**: 기준 전류 60A, 회전 리플 13.3Hz, 기준 속도 800 RPM
- **노이즈**: 가우시안 노이즈 + 간헐적 스파이크

### 끼임 이벤트 유형
| 유형 | 전류 증가 | 속도 감소 | 지속시간 |
|------|----------|----------|----------|
| 경미 (mild) | +30~50% | -15~25% | 0.5~1.0초 |
| 심각 (severe) | +80~150% | -40~70% | 1.5~3.0초 |
| 충격 (shock) | +200~300% | -50~80% | 0.3~0.8초 |

In [ ]:
np.random.seed(42)

# Sampling parameters
DT = 0.01          # 10 ms
T_TOTAL = 300.0    # 300 seconds
N_SAMPLES = int(T_TOTAL / DT)  # 30,000
t = np.arange(N_SAMPLES) * DT

print(f'Sampling period : {DT*1000:.0f} ms')
print(f'Total duration  : {T_TOTAL:.0f} s')
print(f'Total samples   : {N_SAMPLES:,}')

# Base signals
BASE_CUR_A, BASE_CUR_B = 85.0, 60.0
BASE_SPD_A, BASE_SPD_B = 1200.0, 800.0
RIPPLE_FREQ_A, RIPPLE_FREQ_B = 20.0, 13.3

# Current: base + rotation ripple + noise
cur_a = (BASE_CUR_A
         + 3.0 * np.sin(2 * np.pi * RIPPLE_FREQ_A * t)
         + 1.5 * np.sin(2 * np.pi * RIPPLE_FREQ_A * 2 * t + 0.7)
         + np.random.normal(0, 1.2, N_SAMPLES))

cur_b = (BASE_CUR_B
         + 2.5 * np.sin(2 * np.pi * RIPPLE_FREQ_B * t)
         + 1.0 * np.sin(2 * np.pi * RIPPLE_FREQ_B * 2 * t + 1.1)
         + np.random.normal(0, 0.9, N_SAMPLES))

# Speed: base + minor fluctuation + noise
spd_a = BASE_SPD_A + 8.0 * np.sin(2 * np.pi * 0.05 * t) + np.random.normal(0, 3.0, N_SAMPLES)
spd_b = BASE_SPD_B + 5.0 * np.sin(2 * np.pi * 0.03 * t) + np.random.normal(0, 2.0, N_SAMPLES)

# Jamming event definitions
EVENT_TYPES = {
    'mild':   {'cur_gain': (0.30, 0.50), 'spd_drop': (0.15, 0.25), 'dur': (0.5, 1.0)},
    'severe': {'cur_gain': (0.80, 1.50), 'spd_drop': (0.40, 0.70), 'dur': (1.5, 3.0)},
    'shock':  {'cur_gain': (2.00, 3.00), 'spd_drop': (0.50, 0.80), 'dur': (0.3, 0.8)},
}

jam_events = [
    (15.0,  'mild'),
    (38.0,  'severe'),
    (62.0,  'shock'),
    (95.0,  'mild'),
    (118.0, 'severe'),
    (145.0, 'shock'),
    (172.0, 'mild'),
    (198.0, 'severe'),
    (225.0, 'mild'),
    (258.0, 'severe'),
    (280.0, 'shock'),
]

# Ground truth labels (0=normal, 1=mild, 2=severe, 3=shock)
gt_labels = np.zeros(N_SAMPLES, dtype=int)
gt_events = []
INERTIA_DELAY = 0.05  # 50ms speed response delay

for evt_start, evt_type in jam_events:
    params = EVENT_TYPES[evt_type]
    cur_gain = np.random.uniform(*params['cur_gain'])
    spd_drop = np.random.uniform(*params['spd_drop'])
    duration = np.random.uniform(*params['dur'])

    idx_start = int(evt_start / DT)
    idx_end = min(int((evt_start + duration) / DT), N_SAMPLES)
    n_evt = idx_end - idx_start

    # Current spike envelope (ramp up -> sustain -> ramp down)
    ramp_up = min(int(0.1 / DT), n_evt // 3)
    ramp_down = min(int(0.2 / DT), n_evt // 3)
    sustain = n_evt - ramp_up - ramp_down

    envelope = np.concatenate([
        np.linspace(0, 1, ramp_up),
        np.ones(sustain),
        np.linspace(1, 0, ramp_down)
    ])
    if len(envelope) > n_evt:
        envelope = envelope[:n_evt]
    elif len(envelope) < n_evt:
        envelope = np.pad(envelope, (0, n_evt - len(envelope)))

    cur_spike_a = cur_gain * BASE_CUR_A * envelope
    cur_spike_b = cur_gain * BASE_CUR_B * 0.7 * envelope

    if evt_type == 'shock':
        vibration = 5.0 * np.sin(2 * np.pi * 150 * t[idx_start:idx_end])
        cur_spike_a[:len(vibration)] += vibration
        cur_spike_b[:len(vibration)] += vibration * 0.5

    cur_a[idx_start:idx_end] += cur_spike_a
    cur_b[idx_start:idx_end] += cur_spike_b[:n_evt]

    # Speed drop with inertia delay
    delay_samples = int(INERTIA_DELAY / DT)
    spd_idx_start = min(idx_start + delay_samples, N_SAMPLES)
    spd_idx_end = min(idx_end + delay_samples, N_SAMPLES)
    n_spd = spd_idx_end - spd_idx_start

    spd_envelope = np.concatenate([
        np.linspace(0, 1, min(int(0.15 / DT), n_spd // 2)),
        np.ones(max(n_spd - int(0.15 / DT) - int(0.3 / DT), 0)),
        np.linspace(1, 0, min(int(0.3 / DT), n_spd // 2))
    ])[:n_spd]
    if len(spd_envelope) < n_spd:
        spd_envelope = np.pad(spd_envelope, (0, n_spd - len(spd_envelope)))

    spd_a[spd_idx_start:spd_idx_end] -= spd_drop * BASE_SPD_A * spd_envelope
    spd_b[spd_idx_start:spd_idx_end] -= spd_drop * BASE_SPD_B * 0.6 * spd_envelope

    label_val = {'mild': 1, 'severe': 2, 'shock': 3}[evt_type]
    gt_labels[idx_start:idx_end] = label_val
    gt_events.append((idx_start, idx_end, evt_type))

spd_a = np.clip(spd_a, 0, None)
spd_b = np.clip(spd_b, 0, None)

df = pd.DataFrame({
    'time': t, 'CUR_A': cur_a, 'CUR_B': cur_b,
    'SPD_A': spd_a, 'SPD_B': spd_b, 'gt_label': gt_labels,
})

print(f'\n생성된 끼임 이벤트: {len(jam_events)}건')
for i, (s, etype) in enumerate(jam_events):
    si, ei, _ = gt_events[i]
    dur_ms = (ei - si) * DT * 1000
    print(f'  [{i+1:2d}] t={s:6.1f}s | {etype:6s} | {dur_ms:.0f}ms')

print(f'\nDataFrame shape: {df.shape}')
df.head(10)

## Step 2. 데이터 시각화

### 2-1. 전체 300초 오버뷰

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(18, 10), sharex=True)

axes[0].plot(df['time'], df['CUR_A'], lw=0.3, alpha=0.8, label='CUR_A (Main Shaft)')
axes[0].plot(df['time'], df['CUR_B'], lw=0.3, alpha=0.8, label='CUR_B (Sub Shaft)')
axes[0].set_ylabel('Current (A)')
axes[0].set_title('300-Second Overview: Motor Current')
axes[0].legend(loc='upper right')

axes[1].plot(df['time'], df['SPD_A'], lw=0.3, alpha=0.8, label='SPD_A (Main Shaft)')
axes[1].plot(df['time'], df['SPD_B'], lw=0.3, alpha=0.8, label='SPD_B (Sub Shaft)')
axes[1].set_ylabel('Speed (RPM)')
axes[1].set_title('300-Second Overview: Shaft Speed')
axes[1].legend(loc='upper right')

color_map = {1: 'orange', 2: 'red', 3: 'darkred'}
label_map = {1: 'Mild Jam', 2: 'Severe Jam', 3: 'Shock Impact'}
for lbl in [1, 2, 3]:
    mask = df['gt_label'] == lbl
    if mask.any():
        axes[2].fill_between(df['time'], 0, 1, where=mask,
                            color=color_map[lbl], alpha=0.6, label=label_map[lbl])
axes[2].set_ylabel('Event')
axes[2].set_xlabel('Time (s)')
axes[2].set_title('Ground Truth: Jamming Events')
axes[2].set_ylim(0, 1.2)
axes[2].legend(loc='upper right')

plt.tight_layout()
plt.show()

### 2-2. 개별 끼임 이벤트 확대 (유형별 대표 3건)

In [ ]:
repr_events = {}
for si, ei, etype in gt_events:
    if etype not in repr_events:
        repr_events[etype] = (si, ei, etype)

fig, axes = plt.subplots(len(repr_events), 2, figsize=(18, 4 * len(repr_events)))
type_colors = {'mild': 'orange', 'severe': 'red', 'shock': 'darkred'}

for row, (etype, (si, ei, _)) in enumerate(repr_events.items()):
    margin = int(2.0 / DT)
    lo = max(0, si - margin)
    hi = min(N_SAMPLES, ei + margin)
    t_slice = df['time'].values[lo:hi]

    ax_c = axes[row, 0]
    ax_c.plot(t_slice, df['CUR_A'].values[lo:hi], lw=0.5, label='CUR_A')
    ax_c.plot(t_slice, df['CUR_B'].values[lo:hi], lw=0.5, label='CUR_B')
    ax_c.axvspan(si * DT, ei * DT, color=type_colors[etype], alpha=0.2)
    ax_c.set_title(f'Current - {etype.upper()} Event @ t={si*DT:.1f}s')
    ax_c.set_ylabel('Current (A)')
    ax_c.legend(loc='upper right', fontsize=9)

    ax_s = axes[row, 1]
    ax_s.plot(t_slice, df['SPD_A'].values[lo:hi], lw=0.5, label='SPD_A')
    ax_s.plot(t_slice, df['SPD_B'].values[lo:hi], lw=0.5, label='SPD_B')
    ax_s.axvspan(si * DT, ei * DT, color=type_colors[etype], alpha=0.2)
    ax_s.set_title(f'Speed - {etype.upper()} Event @ t={si*DT:.1f}s')
    ax_s.set_ylabel('Speed (RPM)')
    ax_s.legend(loc='upper right', fontsize=9)

axes[-1, 0].set_xlabel('Time (s)')
axes[-1, 1].set_xlabel('Time (s)')
plt.tight_layout()
plt.show()

## Step 3. 패턴 인식 특징 추출기

### 특징 추출 방법
1. **슬라이딩 윈도우 기준선**: 300 샘플(3초) 이동 평균/표준편차
2. **Z-score**: 현재 값이 기준선 대비 얼마나 벗어났는지 표준화
3. **변화율 (dI/dt)**: 전류의 급격한 변화 감지
4. **다중 증거 융합**: 전류 스파이크 + 속도 드롭 + 지속 시간을 종합한 끼임 확률 계산

In [ ]:
class FeatureExtractor:
    def __init__(self, window_size=300, dt=0.01):
        self.W = window_size
        self.dt = dt

    def compute_all(self, cur_a, cur_b, spd_a, spd_b):
        n = len(cur_a)
        cur_a_s = pd.Series(cur_a)
        cur_b_s = pd.Series(cur_b)
        spd_a_s = pd.Series(spd_a)
        spd_b_s = pd.Series(spd_b)

        roll_a = cur_a_s.rolling(self.W, min_periods=50)
        roll_b = cur_b_s.rolling(self.W, min_periods=50)
        roll_sa = spd_a_s.rolling(self.W, min_periods=50)
        roll_sb = spd_b_s.rolling(self.W, min_periods=50)

        base_cur_a = roll_a.mean().values
        base_cur_b = roll_b.mean().values
        std_cur_a = np.maximum(roll_a.std().values, 0.5)
        std_cur_b = np.maximum(roll_b.std().values, 0.5)
        base_spd_a = roll_sa.mean().values
        base_spd_b = roll_sb.mean().values
        std_spd_a = np.maximum(roll_sa.std().values, 1.0)
        std_spd_b = np.maximum(roll_sb.std().values, 1.0)

        # Z-scores
        z_cur_a = (cur_a - base_cur_a) / std_cur_a
        z_cur_b = (cur_b - base_cur_b) / std_cur_b
        z_spd_a = (base_spd_a - spd_a) / std_spd_a  # inverted: drop = positive
        z_spd_b = (base_spd_b - spd_b) / std_spd_b
        z_cur = np.maximum(z_cur_a, z_cur_b)
        z_spd = np.maximum(z_spd_a, z_spd_b)

        # Change rate dI/dt
        diff_step = 5
        di_dt_a = np.zeros(n)
        di_dt_b = np.zeros(n)
        di_dt_a[diff_step:] = (cur_a[diff_step:] - cur_a[:-diff_step]) / (diff_step * self.dt)
        di_dt_b[diff_step:] = (cur_b[diff_step:] - cur_b[:-diff_step]) / (diff_step * self.dt)
        di_dt_smooth = pd.Series(np.maximum(di_dt_a, di_dt_b)).rolling(10, min_periods=1).mean().values

        # Multi-evidence fusion
        def sigmoid(x, center, scale):
            return 1.0 / (1.0 + np.exp(-(x - center) / scale))

        p_cur = sigmoid(z_cur, center=3.0, scale=1.0)
        p_spd = sigmoid(z_spd, center=2.0, scale=0.8)
        p_didt = sigmoid(di_dt_smooth, center=500, scale=200)
        jam_prob = 0.45 * p_cur + 0.30 * p_spd + 0.25 * p_didt

        return {
            'z_cur_a': z_cur_a, 'z_cur_b': z_cur_b,
            'z_spd_a': z_spd_a, 'z_spd_b': z_spd_b,
            'z_cur': z_cur, 'z_spd': z_spd,
            'di_dt_a': di_dt_a, 'di_dt_b': di_dt_b,
            'di_dt_smooth': di_dt_smooth,
            'p_cur': p_cur, 'p_spd': p_spd, 'p_didt': p_didt,
            'jam_prob': jam_prob,
            'base_cur_a': base_cur_a, 'base_cur_b': base_cur_b,
            'base_spd_a': base_spd_a, 'base_spd_b': base_spd_b,
        }

fe = FeatureExtractor(window_size=300, dt=DT)
features = fe.compute_all(
    df['CUR_A'].values, df['CUR_B'].values,
    df['SPD_A'].values, df['SPD_B'].values
)

for key in ['z_cur', 'z_spd', 'di_dt_smooth', 'jam_prob']:
    df[key] = features[key]

print('특징 추출 완료!')
print(f'  Z-score 전류 범위 : [{features["z_cur"][300:].min():.2f}, {features["z_cur"][300:].max():.2f}]')
print(f'  Z-score 속도 범위 : [{features["z_spd"][300:].min():.2f}, {features["z_spd"][300:].max():.2f}]')
print(f'  dI/dt 최대값      : {features["di_dt_smooth"].max():.1f} A/s')
print(f'  끼임확률 범위     : [{features["jam_prob"][300:].min():.4f}, {features["jam_prob"][300:].max():.4f}]')

## Step 4. 상태 머신 기반 감지 알고리즘

### 상태 정의
| 상태 | 설명 | 끼임확률 기준 | 유지 시간 |
|------|------|-------------|----------|
| INIT | 초기화 (기준선 수집) | - | 3초 |
| NORMAL | 정상 운전 | < 0.25 | - |
| CAUTION | 주의 | 0.25 ~ 0.45 | 200ms 이상 |
| WARNING | 경고 | 0.45 ~ 0.65 | 100ms 이상 |
| JAMMING | 끼임 확정 | > 0.65 | 50ms 이상 |

### 히스테리시스 적용
- 상태 상승: 임계값 초과 + 최소 지속시간 충족 시 전이
- 상태 하강: 임계값 - 0.05 (마진) 미만 + 최소 지속시간 충족 시 전이
- 채터링 방지: 상태 전환 후 최소 유지 시간 100ms

In [ ]:
class JammingDetector:
    INIT = 0
    NORMAL = 1
    CAUTION = 2
    WARNING = 3
    JAMMING = 4

    STATE_NAMES = {0: 'INIT', 1: 'NORMAL', 2: 'CAUTION', 3: 'WARNING', 4: 'JAMMING'}

    TH_UP = {2: 0.25, 3: 0.45, 4: 0.65}
    HYSTERESIS = 0.05
    MIN_HOLD_UP = {2: 20, 3: 10, 4: 5}  # samples
    MIN_HOLD_DOWN = 10
    MIN_DWELL = 10

    def __init__(self, init_period=300):
        self.init_period = init_period

    def run(self, jam_prob_array):
        n = len(jam_prob_array)
        states = np.zeros(n, dtype=int)
        state = self.INIT
        hold_counter = 0
        dwell_counter = 0
        candidate_state = None

        for i in range(n):
            if i < self.init_period:
                states[i] = self.INIT
                continue

            if state == self.INIT:
                state = self.NORMAL
                dwell_counter = self.MIN_DWELL

            p = jam_prob_array[i]

            if p >= self.TH_UP[self.JAMMING]:
                target = self.JAMMING
            elif p >= self.TH_UP[self.WARNING]:
                target = self.WARNING
            elif p >= self.TH_UP[self.CAUTION]:
                target = self.CAUTION
            else:
                target = self.NORMAL

            if dwell_counter > 0:
                dwell_counter -= 1
                states[i] = state
                continue

            if target > state:
                if candidate_state == target:
                    hold_counter += 1
                    if hold_counter >= self.MIN_HOLD_UP.get(target, 10):
                        state = target
                        dwell_counter = self.MIN_DWELL
                        candidate_state = None
                        hold_counter = 0
                else:
                    candidate_state = target
                    hold_counter = 1
            elif target < state:
                down_th = self.TH_UP.get(state, 0.25) - self.HYSTERESIS
                if p < down_th:
                    if candidate_state == target:
                        hold_counter += 1
                        if hold_counter >= self.MIN_HOLD_DOWN:
                            state = max(target, self.NORMAL)
                            dwell_counter = self.MIN_DWELL
                            candidate_state = None
                            hold_counter = 0
                    else:
                        candidate_state = target
                        hold_counter = 1
                else:
                    candidate_state = None
                    hold_counter = 0
            else:
                candidate_state = None
                hold_counter = 0

            states[i] = state

        return states


detector = JammingDetector(init_period=300)
det_states = detector.run(features['jam_prob'])
df['det_state'] = det_states

print('=== 감지 결과 요약 ===')
for s_val, s_name in JammingDetector.STATE_NAMES.items():
    count = np.sum(det_states == s_val)
    pct = count / N_SAMPLES * 100
    dur = count * DT
    print(f'  {s_name:8s}: {count:6,} samples ({pct:5.1f}%) = {dur:.1f}s')

## Step 5. 감지 성능 분석

### 평가 지표
- **감지 지연시간 (Detection Latency)**: 실제 이벤트 시작 -> 감지기가 CAUTION 이상 전이까지의 시간
- **오탐(False Positive)**: 정상 구간에서 WARNING 또는 JAMMING 상태로 전이
- **미탐(False Negative)**: 실제 이벤트 구간에서 NORMAL 상태 유지
- **상태 분포**: 전체 시간 대비 각 상태의 비율

In [ ]:
print('=' * 70)
print('감지 성능 분석 (Detection Performance Analysis)')
print('=' * 70)

latencies = []
max_states_reached = []
detected_flags = []

for evt_idx, (si, ei, etype) in enumerate(gt_events):
    search_end = min(ei + int(1.0 / DT), N_SAMPLES)
    detected = False
    latency_ms = None
    max_state = 0

    for j in range(si, search_end):
        if det_states[j] >= JammingDetector.CAUTION:
            latency_ms = (j - si) * DT * 1000
            detected = True
            break

    max_state = det_states[si:ei].max() if ei > si else 0
    latencies.append(latency_ms)
    max_states_reached.append(max_state)
    detected_flags.append(detected)

    status = f'Detected ({latency_ms:.0f}ms)' if detected else 'MISSED'
    max_s_name = JammingDetector.STATE_NAMES[max_state]
    print(f'  Event {evt_idx+1:2d} [{etype:6s}] t={si*DT:6.1f}s : {status:20s} | Max: {max_s_name}')

n_detected = sum(detected_flags)
n_total = len(gt_events)
detected_latencies = [l for l in latencies if l is not None]

print(f'\n--- 요약 ---')
print(f'감지율      : {n_detected}/{n_total} ({n_detected/n_total*100:.1f}%)')
if detected_latencies:
    print(f'평균 지연시간: {np.mean(detected_latencies):.1f} ms')
    print(f'최소 지연시간: {np.min(detected_latencies):.1f} ms')
    print(f'최대 지연시간: {np.max(detected_latencies):.1f} ms')
    print(f'중앙 지연시간: {np.median(detected_latencies):.1f} ms')

# False positive
normal_mask = (gt_labels == 0) & (np.arange(N_SAMPLES) >= 300)
fp_mask = normal_mask & (det_states >= JammingDetector.WARNING)
fp_samples = fp_mask.sum()
fp_duration = fp_samples * DT
normal_total = normal_mask.sum()

print(f'\n--- 오탐 분석 (False Positive) ---')
print(f'정상 구간 총 샘플  : {normal_total:,}')
print(f'오탐 샘플 (>=WARNING): {fp_samples:,} ({fp_samples/max(normal_total,1)*100:.3f}%)')
print(f'오탐 총 시간       : {fp_duration:.2f}s')

# False negative
fn_total = 0
for si, ei, etype in gt_events:
    fn_total += np.sum(det_states[si:ei] <= JammingDetector.NORMAL)

event_total = sum(ei - si for si, ei, _ in gt_events)
print(f'\n--- 미탐 분석 (False Negative) ---')
print(f'이벤트 구간 총 샘플: {event_total:,}')
print(f'미탐 샘플 (NORMAL)  : {fn_total:,} ({fn_total/max(event_total,1)*100:.1f}%)')

## Step 6. 결과 시각화

### 6-1. 전류 신호 + 스파이크 마커

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(18, 7), sharex=True)

ax1.plot(df['time'], df['CUR_A'], lw=0.3, color='steelblue', alpha=0.7, label='CUR_A')
ax1.plot(df['time'], features['base_cur_a'], lw=1.0, color='navy', ls='--', alpha=0.8, label='Baseline A')
type_colors = {'mild': 'orange', 'severe': 'red', 'shock': 'darkred'}
for si, ei, etype in gt_events:
    ax1.axvspan(si * DT, ei * DT, color=type_colors[etype], alpha=0.15)
    ax1.annotate(etype, xy=(si * DT, df['CUR_A'].values[si:ei].max()),
                fontsize=8, color=type_colors[etype], fontweight='bold', ha='left', va='bottom')
ax1.set_ylabel('Current A (A)')
ax1.set_title('Motor Current A with Jamming Event Markers')
ax1.legend(loc='upper right')

ax2.plot(df['time'], df['CUR_B'], lw=0.3, color='coral', alpha=0.7, label='CUR_B')
ax2.plot(df['time'], features['base_cur_b'], lw=1.0, color='darkred', ls='--', alpha=0.8, label='Baseline B')
for si, ei, etype in gt_events:
    ax2.axvspan(si * DT, ei * DT, color=type_colors[etype], alpha=0.15)
ax2.set_ylabel('Current B (A)')
ax2.set_xlabel('Time (s)')
ax2.set_title('Motor Current B with Jamming Event Markers')
ax2.legend(loc='upper right')

plt.tight_layout()
plt.show()

### 6-2. 속도 신호 + 드롭 마커

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(18, 7), sharex=True)

ax1.plot(df['time'], df['SPD_A'], lw=0.3, color='seagreen', alpha=0.7, label='SPD_A')
ax1.plot(df['time'], features['base_spd_a'], lw=1.0, color='darkgreen', ls='--', alpha=0.8, label='Baseline A')
for si, ei, etype in gt_events:
    ax1.axvspan(si * DT, ei * DT, color=type_colors[etype], alpha=0.15)
    min_spd = df['SPD_A'].values[si:ei].min()
    ax1.annotate(f'{etype}\n{min_spd:.0f}RPM', xy=(si * DT, min_spd),
                fontsize=7, color=type_colors[etype], fontweight='bold', ha='left', va='top')
ax1.set_ylabel('Speed A (RPM)')
ax1.set_title('Shaft Speed A with Jamming-Induced Speed Drops')
ax1.legend(loc='lower right')

ax2.plot(df['time'], df['SPD_B'], lw=0.3, color='mediumpurple', alpha=0.7, label='SPD_B')
ax2.plot(df['time'], features['base_spd_b'], lw=1.0, color='indigo', ls='--', alpha=0.8, label='Baseline B')
for si, ei, etype in gt_events:
    ax2.axvspan(si * DT, ei * DT, color=type_colors[etype], alpha=0.15)
ax2.set_ylabel('Speed B (RPM)')
ax2.set_xlabel('Time (s)')
ax2.set_title('Shaft Speed B with Jamming-Induced Speed Drops')
ax2.legend(loc='lower right')

plt.tight_layout()
plt.show()

### 6-3. Z-score 시간 변화

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(18, 7), sharex=True)

ax1.plot(df['time'], features['z_cur'], lw=0.4, color='steelblue', alpha=0.7, label='Z-score (Current)')
ax1.axhline(y=3.0, color='orange', ls='--', lw=1.5, label='Caution Threshold (3.0)')
ax1.axhline(y=5.0, color='red', ls='--', lw=1.5, label='Warning Threshold (5.0)')
for si, ei, etype in gt_events:
    ax1.axvspan(si * DT, ei * DT, color='gray', alpha=0.1)
ax1.set_ylabel('Z-score')
ax1.set_title('Current Z-score Evolution')
ax1.legend(loc='upper right')
ax1.set_ylim(-3, min(features['z_cur'][300:].max() * 1.1, 30))

ax2.plot(df['time'], features['z_spd'], lw=0.4, color='seagreen', alpha=0.7, label='Z-score (Speed Drop)')
ax2.axhline(y=2.0, color='orange', ls='--', lw=1.5, label='Caution Threshold (2.0)')
ax2.axhline(y=4.0, color='red', ls='--', lw=1.5, label='Warning Threshold (4.0)')
for si, ei, etype in gt_events:
    ax2.axvspan(si * DT, ei * DT, color='gray', alpha=0.1)
ax2.set_ylabel('Z-score')
ax2.set_xlabel('Time (s)')
ax2.set_title('Speed Drop Z-score Evolution')
ax2.legend(loc='upper right')
ax2.set_ylim(-3, min(features['z_spd'][300:].max() * 1.1, 30))

plt.tight_layout()
plt.show()

### 6-4. 끼임 확률 (Jamming Probability 0~1)

In [ ]:
fig, ax = plt.subplots(figsize=(18, 5))

ax.plot(df['time'], features['jam_prob'], lw=0.5, color='darkblue', alpha=0.7, label='Jamming Probability')
ax.axhline(y=0.25, color='gold', ls='--', lw=1.5, alpha=0.8, label='CAUTION (0.25)')
ax.axhline(y=0.45, color='orange', ls='--', lw=1.5, alpha=0.8, label='WARNING (0.45)')
ax.axhline(y=0.65, color='red', ls='--', lw=1.5, alpha=0.8, label='JAMMING (0.65)')

ax.fill_between(df['time'], 0, features['jam_prob'],
                where=features['jam_prob'] >= 0.65, color='red', alpha=0.3)
ax.fill_between(df['time'], 0, features['jam_prob'],
                where=(features['jam_prob'] >= 0.45) & (features['jam_prob'] < 0.65),
                color='orange', alpha=0.3)
ax.fill_between(df['time'], 0, features['jam_prob'],
                where=(features['jam_prob'] >= 0.25) & (features['jam_prob'] < 0.45),
                color='gold', alpha=0.2)

for si, ei, etype in gt_events:
    ax.axvspan(si * DT, ei * DT, color='gray', alpha=0.08)

ax.set_ylim(0, 1.05)
ax.set_ylabel('Jamming Probability')
ax.set_xlabel('Time (s)')
ax.set_title('Multi-Evidence Fusion: Jamming Probability (0 ~ 1)')
ax.legend(loc='upper right', ncol=2)
plt.tight_layout()
plt.show()

### 6-5. 이벤트 타임라인 (상태 다이어그램)

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(18, 6), sharex=True)

state_colors = {0: '#cccccc', 1: '#4CAF50', 2: '#FFC107', 3: '#FF9800', 4: '#F44336'}
state_labels = {0: 'INIT', 1: 'NORMAL', 2: 'CAUTION', 3: 'WARNING', 4: 'JAMMING'}

ax1.plot(df['time'], det_states, lw=0.8, color='black', alpha=0.3)
for s_val in range(5):
    mask = det_states == s_val
    if mask.any():
        ax1.fill_between(df['time'], 0, det_states,
                        where=mask, color=state_colors[s_val], alpha=0.6,
                        label=state_labels[s_val])
ax1.set_ylabel('Detection State')
ax1.set_yticks([0, 1, 2, 3, 4])
ax1.set_yticklabels(['INIT', 'NORMAL', 'CAUTION', 'WARNING', 'JAMMING'])
ax1.set_title('State Machine Output: Detection State Timeline')
ax1.legend(loc='upper right', ncol=5, fontsize=9)
ax1.set_ylim(-0.2, 4.5)

gt_state_equiv = np.zeros(N_SAMPLES)
gt_state_equiv[gt_labels == 1] = 2
gt_state_equiv[gt_labels == 2] = 4
gt_state_equiv[gt_labels == 3] = 4

ax2.step(df['time'], gt_state_equiv, lw=1.5, color='blue', alpha=0.5,
         label='Ground Truth (equivalent state)', where='post')
ax2.step(df['time'], det_states, lw=1.0, color='red', alpha=0.7,
         label='Detection Output', where='post')
ax2.set_ylabel('State Level')
ax2.set_xlabel('Time (s)')
ax2.set_yticks([0, 1, 2, 3, 4])
ax2.set_yticklabels(['INIT', 'NORMAL', 'CAUTION', 'WARNING', 'JAMMING'])
ax2.set_title('Ground Truth vs Detection Output')
ax2.legend(loc='upper right')
ax2.set_ylim(-0.2, 4.5)

plt.tight_layout()
plt.show()

### 6-6. 감지 성능: 지연시간 히스토그램 + 이벤트별 분석

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Latency histogram
valid_latencies = [l for l in latencies if l is not None]
if valid_latencies:
    ax = axes[0]
    ax.hist(valid_latencies, bins=15, color='steelblue', edgecolor='white', alpha=0.8)
    ax.axvline(np.mean(valid_latencies), color='red', ls='--', lw=2,
              label=f'Mean: {np.mean(valid_latencies):.0f}ms')
    ax.axvline(np.median(valid_latencies), color='orange', ls='--', lw=2,
              label=f'Median: {np.median(valid_latencies):.0f}ms')
    ax.set_xlabel('Detection Latency (ms)')
    ax.set_ylabel('Count')
    ax.set_title('Detection Latency Distribution')
    ax.legend()

# Latency by event type
ax = axes[1]
type_latencies = {'mild': [], 'severe': [], 'shock': []}
for i, (si, ei, etype) in enumerate(gt_events):
    if latencies[i] is not None:
        type_latencies[etype].append(latencies[i])

type_names = list(type_latencies.keys())
type_means = [np.mean(v) if v else 0 for v in type_latencies.values()]
type_stds = [np.std(v) if len(v) > 1 else 0 for v in type_latencies.values()]
bar_colors = ['#FFC107', '#F44336', '#8B0000']

bars = ax.bar(type_names, type_means, yerr=type_stds, capsize=8,
              color=bar_colors, edgecolor='white', alpha=0.8)
for bar, mean_val in zip(bars, type_means):
    if mean_val > 0:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
               f'{mean_val:.0f}ms', ha='center', va='bottom', fontweight='bold')
ax.set_xlabel('Event Type')
ax.set_ylabel('Mean Latency (ms)')
ax.set_title('Detection Latency by Event Type')

# State distribution pie
ax = axes[2]
state_counts = {}
for s_val, s_name in JammingDetector.STATE_NAMES.items():
    cnt = np.sum(det_states == s_val)
    if cnt > 0:
        state_counts[s_name] = cnt

colors_pie = [state_colors[i] for i, s_name in JammingDetector.STATE_NAMES.items()
              if s_name in state_counts]
ax.pie(state_counts.values(), labels=state_counts.keys(), colors=colors_pie,
       autopct='%1.1f%%', startangle=90, textprops={'fontsize': 10})
ax.set_title('State Distribution (300s)')

plt.tight_layout()
plt.show()

## Step 7. 종합 요약

### 알고리즘 구성 요약

| 구성 요소 | 상세 |
|-----------|------|
| **샘플링** | 10ms (100Hz), 30,000 샘플 |
| **기준선** | 3초 슬라이딩 윈도우 이동 평균/표준편차 |
| **특징 1** | Z-score (전류): 기준선 대비 표준화된 전류 편차 |
| **특징 2** | Z-score (속도): 기준선 대비 속도 드롭 |
| **특징 3** | dI/dt: 5-샘플 차분 기반 전류 변화율 |
| **융합** | 가중 시그모이드 (전류 45% + 속도 30% + dI/dt 25%) |
| **상태 머신** | INIT -> NORMAL -> CAUTION -> WARNING -> JAMMING |
| **히스테리시스** | 상승 임계값 + 0.05 마진, 최소 유지 시간 적용 |

### 핵심 성과
- **다중 증거 융합**으로 단일 특징 기반 대비 오탐율 대폭 감소
- **상태 머신 + 히스테리시스**로 채터링 방지
- **관성 지연 모델링**으로 실제 파쇄기 물리 현상 반영
- 경미/심각/충격 등 **다양한 유형별 차등 감지** 가능

### 실제 적용 시 고려사항
1. **임계값 튜닝**: 현장 데이터 기반으로 Z-score 및 확률 임계값 재조정 필요
2. **샘플링 주기**: PLC 스캔 타임에 맞춰 10~50ms 범위에서 조정
3. **다축 상관**: A/B축 전류 비율 변화도 추가 특징으로 활용 가능
4. **계절/온도 보상**: 주변 온도에 따른 모터 특성 변화 보상 로직 추가 권장

In [ ]:
print('=' * 70)
print('  파쇄기 이물질 끼임 감지 시스템 - 최종 결과')
print('=' * 70)
print(f'\n  데이터: {N_SAMPLES:,} 샘플 ({T_TOTAL:.0f}초, {DT*1000:.0f}ms 간격)')
print(f'  이벤트: {len(gt_events)}건 (경미 {sum(1 for _,_,e in gt_events if e=="mild")}건, '
      f'심각 {sum(1 for _,_,e in gt_events if e=="severe")}건, '
      f'충격 {sum(1 for _,_,e in gt_events if e=="shock")}건)')
print(f'\n  감지율: {n_detected}/{n_total} ({n_detected/n_total*100:.1f}%)')
if detected_latencies:
    print(f'  평균 감지 지연: {np.mean(detected_latencies):.0f}ms')
    print(f'  최소/최대 지연: {np.min(detected_latencies):.0f}ms / {np.max(detected_latencies):.0f}ms')
print(f'  오탐 시간: {fp_duration:.2f}s / {normal_total*DT:.1f}s 정상 구간')
print(f'\n  상태 분포:')
for s_val, s_name in JammingDetector.STATE_NAMES.items():
    cnt = np.sum(det_states == s_val)
    pct = cnt / N_SAMPLES * 100
    print(f'    {s_name:8s}: {pct:5.1f}%')
print('\n' + '=' * 70)